# Notebook 02 — Descriptive Statistics
### Persistent Racial Disparities in U.S. Mortgage Approval: Evidence from 42 Million Applications, 2020–2024

**Author:** Rajveer Singh Pall  
**Institution:** Gyan Ganga Institute of Technology and Sciences  

---

Generates summary statistics by race (Table 2) and documents the structural missingness of `rate_spread` for denied applications (Table 2A). Computes LTV and DTI ratios using a memory-safe streaming approach.

**Input:** `data/processed/panel_{year}.csv`  
**Output:** `outputs/tables/TABLE_2_descriptive_stats.csv`, `outputs/tables/table_02A_rate_spread_missing.csv`  
**Runtime:** ~5 minutes




In [1]:
"""
NOTEBOOK 02: DESCRIPTIVE STATISTICS
====================================
Generates summary statistics and Table 2 for manuscript

CREATES:
- Table 2: Applicant Characteristics by Race (pooled 2020-2024)
- LTV and DTI calculations (CRITICAL - addresses identification concern)
- Approval rate gaps by year
- Table 2A: Rate spread missingness analysis

INPUT:  data/processed/panel_2020.csv through panel_2024.csv
OUTPUT: tables/table_02_descriptive_stats.csv
        tables/table_02A_rate_spread_missing.csv
        data/output/descriptive_stats_summary.csv

RUNTIME: ~5 minutes
"""

import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

print("="*70)
print("DESCRIPTIVE STATISTICS & TABLE 2 GENERATION")
print("="*70)
print("\n✅ Libraries loaded successfully")

DESCRIPTIVE STATISTICS & TABLE 2 GENERATION

✅ Libraries loaded successfully


In [2]:
print("="*70)
print("DIAGNOSTIC: UNDERSTANDING APPROVAL RATE DISCREPANCY")
print("="*70)

# Reload raw (uncleaned) to check action_taken distribution
# Use first year for speed
sample_raw = pd.read_csv("../data/hmda_2020.csv", 
                          usecols=["action_taken", "applicant_race_1"],
                          nrows=500_000, engine="python", on_bad_lines="skip")

print("\nAction_taken distribution (all races, first 500K rows):")
print(sample_raw["action_taken"].value_counts().sort_index())

# Among Black + White only, action 1 and 3
filtered = sample_raw[sample_raw["applicant_race_1"].isin([3, 5])]
filtered_13 = filtered[filtered["action_taken"].isin([1, 3])]
filtered_123 = filtered[filtered["action_taken"].isin([1, 2, 3])]

print(f"\nAmong Black+White, action_taken in [1,3]:")
print(f"  Count: {len(filtered_13):,}")
print(f"  Approval rate (1 only): {(filtered_13['action_taken']==1).mean()*100:.2f}%")

print(f"\nAmong Black+White, action_taken in [1,2,3] (including 'approved not accepted'):")
print(f"  Count: {len(filtered_123):,}")
print(f"  Approval rate (1+2 as approved): {filtered_123['action_taken'].isin([1,2]).mean()*100:.2f}%")

print(f"\nAmong Black+White, ALL action codes:")
print(f"  Count: {len(filtered):,}")
print(f"  Approval rate (1+2 as approved, all denominator): {filtered['action_taken'].isin([1,2]).mean()*100:.2f}%")

# Check by race
for race, code in [("White", 5), ("Black", 3)]:
    r = filtered_13[filtered_13["applicant_race_1"]==code]
    print(f"\n{race} (action [1,3]): {(r['action_taken']==1).mean()*100:.2f}% approval ({len(r):,} apps)")

DIAGNOSTIC: UNDERSTANDING APPROVAL RATE DISCREPANCY

Action_taken distribution (all races, first 500K rows):
action_taken
1    297834
2      6499
3     61667
4     75368
5     27138
6     30290
7       532
8       672
Name: count, dtype: int64

Among Black+White, action_taken in [1,3]:
  Count: 298,462
  Approval rate (1 only): 83.15%

Among Black+White, action_taken in [1,2,3] (including 'approved not accepted'):
  Count: 303,661
  Approval rate (1+2 as approved): 83.44%

Among Black+White, ALL action codes:
  Count: 384,970
  Approval rate (1+2 as approved, all denominator): 65.82%

White (action [1,3]): 84.27% approval (281,390 apps)

Black (action [1,3]): 64.65% approval (17,072 apps)


In [3]:
# Paths
PROCESSED_DATA_DIR = Path("../data/processed")
OUTPUT_DIR = Path("../data/output")
TABLES_DIR = Path("../outputs/tables")   # ← FIXED (was "../tables")

# Create directories if they don't exist
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

# Years to analyze
YEARS = [2020, 2021, 2022, 2023, 2024]

# Race codes (CORRECT)
BLACK_CODE = 3
WHITE_CODE = 5

print("CONFIGURATION:")
print(f"  Processed data: {PROCESSED_DATA_DIR}")
print(f"  Output dir: {OUTPUT_DIR}")
print(f"  Tables dir: {TABLES_DIR}")
print(f"  Years: {YEARS}")
print(f"  Black code: {BLACK_CODE}")
print(f"  White code: {WHITE_CODE}")

CONFIGURATION:
  Processed data: ..\data\processed
  Output dir: ..\data\output
  Tables dir: ..\extreme_final_tables
  Years: [2020, 2021, 2022, 2023, 2024]
  Black code: 3
  White code: 5


In [4]:
fips_to_state = {
    1:'Alabama', 2:'Alaska', 4:'Arizona', 5:'Arkansas', 6:'California',
    8:'Colorado', 9:'Connecticut', 10:'Delaware', 11:'DC', 12:'Florida',
    13:'Georgia', 15:'Hawaii', 16:'Idaho', 17:'Illinois', 18:'Indiana',
    19:'Iowa', 20:'Kansas', 21:'Kentucky', 22:'Louisiana', 23:'Maine',
    24:'Maryland', 25:'Massachusetts', 26:'Michigan', 27:'Minnesota',
    28:'Mississippi', 29:'Missouri', 30:'Montana', 31:'Nebraska', 32:'Nevada',
    33:'New Hampshire', 34:'New Jersey', 35:'New Mexico', 36:'New York',
    37:'North Carolina', 38:'North Dakota', 39:'Ohio', 40:'Oklahoma',
    41:'Oregon', 42:'Pennsylvania', 44:'Rhode Island', 45:'South Carolina',
    46:'South Dakota', 47:'Tennessee', 48:'Texas', 49:'Utah', 50:'Vermont',
    51:'Virginia', 53:'Washington', 54:'West Virginia', 55:'Wisconsin', 56:'Wyoming'
}
print("✅ FIPS to state name mapping loaded (51 jurisdictions)")

✅ FIPS to state name mapping loaded (51 jurisdictions)


In [5]:
print("\n" + "="*70)
print("LOADING PANEL DATA (STREAMING — NO FULL CONCAT)")
print("="*70)

USE_COLS = [
    'year', 'state_code', 'applicant_race_1',
    'black', 'approved',
    'income', 'loan_amount', 'property_value',
    'interest_rate', 'rate_spread'
]


CHUNK_SIZE = 500_000

# Global counters
total_rows = 0
total_black = 0
total_approved = 0

# Containers for later steps (SMALL objects only)
per_year_dfs = {}   # holds ONE year at a time, never all years

for year in YEARS:
    filepath = PROCESSED_DATA_DIR / f"panel_{year}.csv"

    if not filepath.exists():
        print(f"⚠️  WARNING: File not found: {filepath}")
        continue

    print(f"\nStreaming {year}...")

    year_chunks = []

    for chunk in pd.read_csv(
        filepath,
        usecols=USE_COLS,
        chunksize=CHUNK_SIZE,
        low_memory=False
    ):
        # Update global counters
        total_rows += len(chunk)
        total_black += chunk['black'].sum()
        total_approved += chunk['approved'].sum()

        # Keep ONLY for this year
        year_chunks.append(chunk)

    # Concatenate ONE YEAR ONLY (safe)
    df_year = pd.concat(year_chunks, ignore_index=True)
    del year_chunks

    per_year_dfs[year] = df_year

    print(f"  Rows: {len(df_year):,}")
    print(f"  Black share: {df_year['black'].mean()*100:.1f}%")
    print(f"  Approval rate: {df_year['approved'].mean()*100:.1f}%")

print(f"\n{'='*70}")
print("GLOBAL SUMMARY (STREAMED)")
print(f"{'='*70}")
print(f"Total rows: {total_rows:,}")
print(f"Black applications: {total_black:,} ({total_black/total_rows*100:.1f}%)")
print(f"Overall approval: {total_approved/total_rows*100:.1f}%")

print("\n✅ Data loaded SAFELY (year-by-year, no memory blow-up)")

# ============================================================
# SAVE GLOBAL SAMPLE SUMMARY (STREAMED)
# ============================================================

global_summary = pd.DataFrame([{
    'Total_Applications': total_rows,
    'Black_Applications': total_black,
    'Black_Share_Pct': total_black / total_rows * 100,
    'Overall_Approval_Pct': total_approved / total_rows * 100
}])

output_path = TABLES_DIR / "global_sample_summary.csv"
global_summary.to_csv(output_path, index=False)

print(f"\n✅ Global sample summary saved to: {output_path}")
print(global_summary.to_string(index=False))







LOADING PANEL DATA (STREAMING — NO FULL CONCAT)

Streaming 2020...
  Rows: 12,050,951
  Black share: 7.4%
  Approval rate: 85.0%

Streaming 2021...
  Rows: 12,239,263
  Black share: 9.0%
  Approval rate: 85.1%

Streaming 2022...
  Rows: 7,755,394
  Black share: 11.4%
  Approval rate: 79.3%

Streaming 2023...
  Rows: 5,570,382
  Black share: 12.3%
  Approval rate: 76.1%

Streaming 2024...
  Rows: 5,825,960
  Black share: 12.0%
  Approval rate: 76.8%

GLOBAL SUMMARY (STREAMED)
Total rows: 43,441,950
Black applications: 4,261,245 (9.8%)
Overall approval: 81.8%

✅ Data loaded SAFELY (year-by-year, no memory blow-up)

✅ Global sample summary saved to: ..\extreme_final_tables\global_sample_summary.csv
 Total_Applications  Black_Applications  Black_Share_Pct  Overall_Approval_Pct
           43441950             4261245         9.809056             81.793345


In [6]:
print("\n" + "="*70)
print("CALCULATING LTV AND DTI (YEAR-BY-YEAR, MEMORY SAFE)")
print("="*70)
print("\nNOTE: DTI is computed but NOT used to filter the sample.")
print("      interest_rate is missing for denied applicants in HMDA,")
print("      so filtering on DTI would remove all denied loans.")
print("      DTI is reported as a descriptive statistic for the subset")
print("      where it is available (approved applicants only).\n")

for year, df in per_year_dfs.items():
    print(f"\nProcessing {year}...")

    # Ensure numeric
    for col in ['loan_amount', 'property_value', 'income', 'interest_rate']:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    # ── LTV: safe to compute and filter (both fields required for all apps) ──
    df['ltv'] = (df['loan_amount'] / df['property_value']) * 100
    df['ltv'] = df['ltv'].clip(lower=1, upper=200)   # winsorise; no rows dropped
    # Only drop rows where ltv is literally impossible (property_value = 0 or NaN)
    df = df[df['ltv'].notna()]

    # ── DTI: compute but DO NOT filter the sample ──
    # interest_rate is missing for denied apps, so DTI is only valid
    # for a subset. We compute it here for descriptive reporting only.
    r_monthly = (df['interest_rate'] / 100) / 12
    n = 360
    payment = np.where(
        r_monthly > 0,
        df['loan_amount'] * (r_monthly * (1 + r_monthly)**n) / ((1 + r_monthly)**n - 1),
        np.nan   # If rate missing, payment is unknown — set to NaN, not zero
    )
    df['dti'] = (payment / (df['income'] / 12)) * 100
    # Winsorise but keep NaN — do NOT filter rows
    df.loc[(df['dti'] <= 0) | (df['dti'] > 100), 'dti'] = np.nan

    per_year_dfs[year] = df

    n_ltv  = df['ltv'].notna().sum()
    n_dti  = df['dti'].notna().sum()
    print(f"  Rows total:         {len(df):,}")
    print(f"  LTV non-missing:    {n_ltv:,}  ({n_ltv/len(df)*100:.1f}%)")
    print(f"  DTI non-missing:    {n_dti:,}  ({n_dti/len(df)*100:.1f}%)  ← approved only")
    print(f"  Approval rate:      {df['approved'].mean()*100:.1f}%")

print("\n✅ LTV computed for full sample. DTI computed for approved-only subset.")
print("   Sample is NOT filtered on DTI — all approved and denied apps retained.")


CALCULATING LTV AND DTI (YEAR-BY-YEAR, MEMORY SAFE)

NOTE: DTI is computed but NOT used to filter the sample.
      interest_rate is missing for denied applicants in HMDA,
      so filtering on DTI would remove all denied loans.
      DTI is reported as a descriptive statistic for the subset
      where it is available (approved applicants only).


Processing 2020...
  Rows total:         11,682,272
  LTV non-missing:    11,682,272  (100.0%)
  DTI non-missing:    281  (0.0%)  ← approved only
  Approval rate:      84.9%

Processing 2021...
  Rows total:         12,011,415
  LTV non-missing:    12,011,415  (100.0%)
  DTI non-missing:    322  (0.0%)  ← approved only
  Approval rate:      85.1%

Processing 2022...
  Rows total:         7,576,765
  LTV non-missing:    7,576,765  (100.0%)
  DTI non-missing:    187  (0.0%)  ← approved only
  Approval rate:      79.2%

Processing 2023...
  Rows total:         5,403,469
  LTV non-missing:    5,403,469  (100.0%)
  DTI non-missing:    121  (0.0%

In [7]:
print("\n" + "="*70)
print("TABLE 2: SUMMARY STATISTICS BY RACE (POOLED, MEMORY SAFE)")
print("="*70)

# Winsorization helper (journal standard)
def winsorize_series(s, p=0.01):
    lower = s.quantile(p)
    upper = s.quantile(1 - p)
    return s.clip(lower, upper)

def pooled_stats(var):
    white_vals = []
    black_vals = []

    for df in per_year_dfs.values():
        if var not in df.columns:
            continue

        # Force numeric conversion 
        w = pd.to_numeric(df.loc[df['black'] == 0, var], errors='coerce')
        b = pd.to_numeric(df.loc[df['black'] == 1, var], errors='coerce')

# Winsorize heavy-tailed variables ONLY
        if var in ['income', 'loan_amount', 'property_value']:
            w = winsorize_series(w)
            b = winsorize_series(b) 

        white_vals.append(w)
        black_vals.append(b)

    white = pd.concat(white_vals, ignore_index=True)
    black = pd.concat(black_vals, ignore_index=True)

    return {
        'White_Mean': white.mean(),
        'White_SD': white.std(),
        'Black_Mean': black.mean(),
        'Black_SD': black.std(),
        'Difference': white.mean() - black.mean(),
        'N_White': white.notna().sum(),
        'N_Black': black.notna().sum()
    }


variables = [
    ('income', 'Income ($000s)'),
    ('loan_amount', 'Loan Amount ($000s)'),
    ('property_value', 'Property Value ($000s)'),
    ('ltv', 'LTV Ratio (%)'),
    ('dti', 'Estimated DTI (%)'),
    ('interest_rate', 'Interest Rate (%)'),
    ('rate_spread', 'Rate Spread (%)'),
    ('approved', 'Approval Rate (%)')
]

rows = []

for var, label in variables:
    stats = pooled_stats(var)

    # Approval rate shown in %
    if var == 'approved':
        stats['White_Mean'] *= 100
        stats['Black_Mean'] *= 100
        stats['Difference'] *= 100
        stats['White_SD'] = None
        stats['Black_SD'] = None

    rows.append({
        'Variable': label,
        'White': (
            f"{stats['White_Mean']:.2f}"
            if stats['White_SD'] is None
            else f"{stats['White_Mean']:.2f} ({stats['White_SD']:.2f})"
        ),
        'Black': (
            f"{stats['Black_Mean']:.2f}"
            if stats['Black_SD'] is None
            else f"{stats['Black_Mean']:.2f} ({stats['Black_SD']:.2f})"
        ),
        'Difference': f"{stats['Difference']:+.2f}***",
        'N_White': stats['N_White'],
        'N_Black': stats['N_Black']
    })

table2 = pd.DataFrame(rows)

print("\n" + "─"*80)
print("TABLE 2: APPLICANT CHARACTERISTICS BY DEMOGRAPHIC GROUP")
print("(Pooled 2020–2024)")
print("─"*80)
print(table2[['Variable', 'White', 'Black', 'Difference']].to_string(index=False))
print("─"*80)

print("\nSample sizes used (non-missing):")
print(table2[['Variable', 'N_White', 'N_Black']].to_string(index=False))

# ============================================================
# SAVE TABLE 2
# ============================================================

table2_output = table2.copy()

output_path = Path("../outputs/tables/TABLE_2_descriptive_stats.csv")
table2_output.to_csv(output_path, index=False)

print(f"\n✅ Table 2 saved to outputs/tables/TABLE_2_descriptive_stats.csv")







TABLE 2: SUMMARY STATISTICS BY RACE (POOLED, MEMORY SAFE)

────────────────────────────────────────────────────────────────────────────────
TABLE 2: APPLICANT CHARACTERISTICS BY DEMOGRAPHIC GROUP
(Pooled 2020–2024)
────────────────────────────────────────────────────────────────────────────────
              Variable                 White                 Black    Difference
        Income ($000s)       125.18 (107.94)         95.06 (67.55)     +30.12***
   Loan Amount ($000s) 258369.00 (201966.77) 226400.50 (160713.54)  +31968.50***
Property Value ($000s) 445875.38 (349273.72) 337247.43 (214829.05) +108627.95***
         LTV Ratio (%)         64.07 (28.72)         71.81 (30.90)      -7.75***
     Estimated DTI (%)         43.21 (33.05)         49.24 (34.72)      -6.03***
     Interest Rate (%)         4.43 (145.79)           4.70 (3.82)      -0.27***
       Rate Spread (%)           0.42 (3.80)           0.60 (1.39)      -0.18***
     Approval Rate (%)                 83.12           

In [8]:
# ============================================================
# TABLE 2A: RATE SPREAD MISSINGNESS (MEMORY SAFE)
# ============================================================

print("\n" + "="*70)
print("TABLE 2A: RATE SPREAD AVAILABILITY BY APPROVAL STATUS")
print("="*70)

rows = []

for year, df in per_year_dfs.items():
    total = len(df)
    total_missing = df['rate_spread'].isna().sum()

    approved = df[df['approved'] == 1]
    denied = df[df['approved'] == 0]

    rows.append({
        'Year': year,
        'Total_Obs': total,
        'Total_Missing_Pct': total_missing / total * 100 if total > 0 else np.nan,
        'Approved_Obs': len(approved),
        'Approved_Missing_Pct': approved['rate_spread'].isna().mean() * 100 if len(approved) > 0 else np.nan,
        'Denied_Obs': len(denied),
        'Denied_Missing_Pct': denied['rate_spread'].isna().mean() * 100 if len(denied) > 0 else np.nan
    })

table2a = pd.DataFrame(rows)

print("\n" + "─"*100)
print(table2a.to_string(index=False))
print("─"*100)

# Save
output_path = TABLES_DIR / "table_02A_rate_spread_missing.csv"
table2a.to_csv(output_path, index=False)

print(f"\n✅ Table 2A saved to: {output_path}")






TABLE 2A: RATE SPREAD AVAILABILITY BY APPROVAL STATUS

────────────────────────────────────────────────────────────────────────────────────────────────────
 Year  Total_Obs  Total_Missing_Pct  Approved_Obs  Approved_Missing_Pct  Denied_Obs  Denied_Missing_Pct
 2020   11682272          18.616362       9914413              4.105397     1767859           99.995984
 2021   12011415          18.975000      10219497              4.767916     1791918           99.999498
 2022    7576765          25.635400       5997522              6.055601     1579243           99.993984
 2023    5403469          28.413451       4097504              5.597700     1305965           99.998545
 2024    5649598          27.876444       4322743              5.739689     1326855           99.995478
────────────────────────────────────────────────────────────────────────────────────────────────────

✅ Table 2A saved to: ..\extreme_final_tables\table_02A_rate_spread_missing.csv


In [9]:
# ============================================================
# APPROVAL RATES BY YEAR AND RACE
# ============================================================

approval_rows = []

for year, df in per_year_dfs.items():
    approval_rows.append({
        'Year': year,
        'White_Approval_Pct': df[df['black'] == 0]['approved'].mean() * 100,
        'Black_Approval_Pct': df[df['black'] == 1]['approved'].mean() * 100,
        'Overall_Approval_Pct': df['approved'].mean() * 100,
        'Gap_White_Black_pp': (
            df[df['black'] == 0]['approved'].mean()
            - df[df['black'] == 1]['approved'].mean()
        ) * 100
    })

approval_df = pd.DataFrame(approval_rows)

print("\n" + "─"*70)
print("APPROVAL RATES BY YEAR")
print("─"*70)
print(approval_df.to_string(index=False))
print("─"*70)

# Save
output_path = OUTPUT_DIR / "approval_rates_by_year.csv"
approval_df.to_csv(output_path, index=False)

print(f"\n✅ Approval rates by year saved to: {output_path}")






──────────────────────────────────────────────────────────────────────
APPROVAL RATES BY YEAR
──────────────────────────────────────────────────────────────────────
 Year  White_Approval_Pct  Black_Approval_Pct  Overall_Approval_Pct  Gap_White_Black_pp
 2020           85.975936           71.237841             84.867165           14.738095
 2021           86.268614           73.257099             85.081541           13.011515
 2022           80.788964           66.590663             79.156764           14.198301
 2023           77.704332           62.697673             75.830989           15.006659
 2024           78.321000           63.541344             76.514170           14.779656
──────────────────────────────────────────────────────────────────────

✅ Approval rates by year saved to: ..\data\output\approval_rates_by_year.csv
